# Google Play Store Apps Analysis

This notebook analyzes Google Play Store app data using Python, pandas, NumPy, Matplotlib, Seaborn, and SciPy.

## Analysis workflow
1. Load and inspect the dataset
2. Explore data types and descriptive statistics
3. Clean and transform app size, installs, and price
4. Investigate missing values and duplicates
5. Explore correlations and relationships between variables
6. Analyze ratings, reviews, installs, and app categories
7. Visualize important patterns and distributions

> **Note:** The dataset is expected at `../data/Play Store Data.csv` when this notebook is opened from the `notebooks/` folder.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline


import warnings 
warnings.filterwarnings('ignore')

## 2. Load the Dataset

In [ ]:
df = pd.read_csv("../data/Play Store Data.csv")
df.head(4)

In [ ]:
df.iloc[10474: 10494]


In [ ]:
df.sample(10)


In [ ]:
df.tail()


## 3. Initial Data Exploration

In [ ]:
# Set the option maximum of rows and column
pd.set_option('display.max_columns', None)

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
# Checking the shape of the columns
print(f'The number of Rows are "{df.shape[0]}", and the number of columns are "{df.shape[1]}"')

In [ ]:
print(f'The name of the columns are: {df.columns}')

In [ ]:
# Checking the info of the dataset
df.info()

In [ ]:
df.describe()

In [ ]:
df.drop(10472, axis=0, inplace=True)

In [ ]:
df.info()

In [ ]:
df['Reviews'] = df['Reviews'].astype('int')

In [ ]:
df.describe()

## 4. Data Cleaning and Feature Transformation

In [ ]:
# Taking size column and make it numeric
df['Size'].value_counts()

In [ ]:
df['Size'].isnull().sum()

In [ ]:
# Checking the number of values in three different categories in Size
print("Number of M in Size Column", df['Size'].loc[df['Size'].str.contains('M')].value_counts().sum())
print("Number of k in Size Column", df['Size'].loc[df['Size'].str.contains('k')].value_counts().sum())
print("Number of Varies with device in Size Column", df['Size'].loc[df['Size'].str.contains('Varies with device')].value_counts().sum())


In [ ]:
# Convert the whole size of the column into bytes
### Defining a Function

def convert_into_bytes(column_name):
    if isinstance(column_name, str):
        if 'k' in column_name:
            return float(column_name.replace("k", "")) * 1024
        elif 'M' in column_name:
            return float(column_name.replace("M", "")) * 1024 * 1024
        elif 'Varies with device' in column_name:
            return np.nan
    return column_name

In [ ]:
df['Size'] = df['Size'].apply(convert_into_bytes)

In [ ]:
df['Size']

In [ ]:
df.rename(columns={'Size': 'Size_in_bytes'}, inplace=True)

In [ ]:
### Convert bytes into mega_bytes

df['Size_MB'] = df['Size_in_bytes'].apply(lambda x : x/(1024* 1024))

In [ ]:
df['Size_MB']

In [ ]:
df['Installs']

In [ ]:
# Remove + sign
# Remove , from the values
# Convert the column in to integers

In [ ]:
## Define a function to deal with installs column

def installs(install):
    if isinstance(install, str):
        if '+' in install:
            return install.replace("+", "")
    return int(install)

In [ ]:
df['Installs'] = df['Installs'].apply(installs)

In [ ]:
df['Installs'] = df['Installs'].apply(lambda x: x.replace(',', '') if ',' in str(x) else x)

In [ ]:
df['Installs'] = df['Installs'].astype('int')

In [ ]:
df['Installs'].value_counts()

In [ ]:
# making a new column called 'Installs_category' which will have the category of the installs
bins = [-1, 0, 10, 1000, 10000, 100000, 1000000, 10000000, 10000000000]
labels=['no', 'Very low', 'Low', 'Moderate', 'More than moderate', 'High', 'Very High', 'Top Notch']
df['Installs_category'] = pd.cut(df['Installs'], bins=bins, labels=labels)

In [ ]:
df['Installs_category'].value_counts()

In [ ]:
df.head(4)

In [ ]:
# Taking Price column
df['Price'].unique()

In [ ]:
def adjust_price(price):
    if isinstance(price, str):
        if '$' in price:
            return price.replace("$", "")
    return price

In [ ]:
df['Price'] = df['Price'].apply(adjust_price)

In [ ]:
df['Price'].unique()

In [ ]:
df['Price'].dtype


In [ ]:
df['Price'] = df['Price'].astype('float')

In [ ]:
df.describe()

In [ ]:
# Observations:
# Now, we have only 6 columns as numeric data type.
# We can observe their descriptive statistics. and make tons of observations as per our hypotheses.
# We can see that the Rating column has a minimum value of 1 and a maximum value of 5, which is the range of rating, and the mean is 4.19 which is a good rating. On an average people give this rating.
# We can see that the Reviews column has a minimum value of 0 and a maximum value of 78,158,306 78+ Millions, which is the range of reviews, and the mean is 444,111.93 which is a good number of reviews. On an average people give this number of reviews to the apps. But it does not make sense to us, as we have different categories of apps.
# Similarly, we can observe the other columns as well.
# Therefore, the most important thing is to classify as app based on the correlation matrix and then observe the descriptive statistics of the app category and number of installs, reviews, ratings, etc.

# But even before that we have to think about the missing values in the dataset.

In [ ]:
df.head()

## 5. Missing-Value Analysis

In [ ]:
# Missing Values
df.isnull().sum().sort_values(ascending=False)

In [ ]:
### Plot Missing Values
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')

In [ ]:
# make figure size
plt.figure(figsize=(16, 6))
# plot the null values by their percentage in each column
missing_percentage = df.isnull().sum()/len(df)*100
missing_percentage.plot(kind='bar')
# add the labels
plt.xlabel('Columns')
plt.ylabel('Percentage')
plt.title('Percentage of Missing Values in each Column')

In [ ]:
#plt.figure(figsize=(16, 6)) # make figure size
missing_percentage[missing_percentage < 1].plot(kind='bar') # plot the null values by their percentage in each column
plt.xlabel('Columns') # add the x-axis labels
plt.ylabel('Percentage') # add the labels for y-axis
plt.title('Percentage of Missing Values in each Column without Rating column')  # add the title for the plot

In [ ]:
# # Observations:
# We have 1695 missing values in the 'Size_in_bytes' and 'Size_in_Mb' columns, which is 15.6% of the total values in the column.
# We have 1474 missing values in the 'Rating' column, which is 13.6% of the total values in the column.
# We have 8 missing value in the 'Current Ver' column, which is 0.07% of the total values in the column.
# We have 2 missing values in the 'Android Ver' column, which is 0.01% of the total values in the column.
# We have only 1 missing value in Category, Type and Genres columns, which is 0.009% of the total values in the column.


## 6. Missing-Value Investigation

In [ ]:
# Dealing with the missing values
# We can not impute the Rating column as is is directly linked with the installation column. To test this Hypothesis we need to plot the Rating column with the Installs and Size columns and statistically test it using pearson correlation test.

In [ ]:
df.columns


In [ ]:
numeric_cols = [i for i in df.columns if df[i].dtype != 'object' ] # make a list of numeric columns

In [ ]:
numeric_cols.remove("Installs_category")

In [ ]:
numeric_cols

## 7. Correlation Analysis

In [ ]:
corr = df[numeric_cols].corr()

In [ ]:
corr

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(corr, cmap=sns.diverging_palette(220, 20, as_cmap=True))
plt.show()

In [ ]:
# we can calculate the pearson correlation coefficient using scipy as well as follows
# this is to install scipy if you have not done it before
# pip install scipy 
from scipy import stats

# remove rows containing NaN or infinite values (Important to calculate Pearson's R)
df_clean = df.dropna()

# calculate Pearson's R between Rating and Installs
pearson_r, _ = stats.pearsonr(df_clean['Reviews'], df_clean['Installs'])
print(f"Pearson's R between Reviews and Installs: {pearson_r:.4f}")

In [ ]:
# Observations
# Lighter color shows the high correlation and darker color shows the low correlation
# We can see that the Reviews column has a high correlation with the Installs column, which is 0.64 according to corr(). Which is quite good.
# This shows that the more the reviews the more the installs are for one app. If in any case we need to impute reviews we have to think of number of install.
# If we have an ap with 2 installs and we imputer the reviews with 1000 or via average reviews then it will be wrong.
# Installs is slightly correlated with Size_in_Mb or Size_in_bytes , which is 0.16, this also shows us the importance of size and Installs. But we can not depend on it as the Peason correlation is very low.

In [ ]:
# Before going ahead, let's remove the rows with missing values in the Current Ver, Android Ver, Category, Type and Genres columns, as they are very less in number and will not affect our analysis.

In [ ]:
# remove the rows having null values in the 'Current Ver', 'Android Ver', 'Category', 'Type' and 'Genres' column
df.dropna(subset=['Current Ver', 'Android Ver', 'Category', 'Type', 'Genres'], inplace=True)

In [ ]:
# length after removing null values
print(f"Length of the dataframe after removing null values: {len(df)}")

In [ ]:
# # Observations
# Only Rating and Size_in_bytes or Size_in_Mb columns are left with missing values.
# We know that we have to be carefull while deadling with Rating column, as it is directly linked with the Installs column.
# In Size columns we already know about Varies with device values, which we have converted into null values, we do not need to impute at the moment, as every app has different size and nobody can predict that as nearly as possible.


## 8. Ratings, Reviews, and Installs Analysis

In [ ]:
# use groupby function to find the trend of Rating in each Installs_category
df.groupby('Installs_category')['Rating'].describe()

In [ ]:
df['Rating'].isnull().sum()

In [ ]:
# in which Install_category the Rating has NaN values
df['Installs_category'].loc[df['Rating'].isnull()].value_counts()

In [ ]:
# plot the boxplot of Rating in each Installs_category
plt.figure(figsize=(16, 6)) # make figure size
sns.boxplot(x='Installs_category', y='Rating', hue='Installs_category', data=df) # plot the boxplot
# add the text of number of null values in each category
plt.text(0, 3.5, 'Null values: 14')
plt.text(1, 3.5, 'Null values: 874')
plt.text(2, 3.5, 'Null values: 86')
plt.text(3, 3.5, 'Null values: 31')
plt.text(4, 3.5, 'Null values: 3')
plt.text(5, 3.5, 'Null values: 0')
plt.text(6, 3.5, 'Null values: 0')
plt.text(7, 3.5, 'Null values: 0')

In [ ]:
def fill_missing_ratings(df, category, fill_value):
  """Fills missing rating values in a specified category with a given value.

  Args:
    df: The pandas DataFrame containing the data.
    category: The category to fill missing values for.
    fill_value: The value to fill missing ratings with.

  Returns:
    The modified DataFrame with filled missing values.
  """

  # Filter the DataFrame for rows where the category matches and rating is missing
  filtered_df = df[(df['Installs_category'] == category) & df['Rating'].isnull()]

  # Fill the missing values with the specified value
  df.loc[filtered_df.index, 'Rating'] = fill_value

  return df

In [ ]:
df = fill_missing_ratings(df, 'Low', 4.170970)

In [ ]:
df = fill_missing_ratings(df, 'Very low', 4.637037)
df = fill_missing_ratings(df, 'Moderate', 4.035417)
df = fill_missing_ratings(df, 'More than moderate', 4.093255)
df = fill_missing_ratings(df, 'High', 4.207525)

In [ ]:
df = fill_missing_ratings(df, 'no', 0)

In [ ]:
# in which Install_category the Rating has NaN values
df['Installs_category'].loc[df['Rating'].isnull()].value_counts()

In [ ]:
df['Rating'].isnull().sum()

In [ ]:
# let's plot the same plots for Reviews column as well
plt.figure(figsize=(16, 6)) # make figure size
sns.boxplot(x='Installs_category', y= 'Reviews', data=df) # plot the boxplot

In [ ]:
# let's plot the same plots for Reviews column as well
plt.figure(figsize=(16, 6)) # make figure size
sns.boxplot(x='Installs_category', y= np.log10(df['Reviews']), data=df) # plot the boxplot

In [ ]:
# Draw a scatter plot between Rating, Reviews and Installs
plt.figure(figsize=(16, 6)) # make figure size
sns.scatterplot(x='Rating', y='Reviews', hue='Installs_category', data=df) # plot the scatter plot

In [ ]:
# It doesn't show any trend, because, you should know that Rating is a categorical variable (Ordinal) and Reviews is a continuous variable, therefore, we can not plot them together.
# Let's try with Reviews and Installs

In [ ]:
# plot reviews and installs in a scatter plot
plt.figure(figsize=(16, 6)) # make figure size
sns.scatterplot(x='Reviews', y='Installs', data=df) # plot the scatter plot

In [ ]:
# We did not see any trend and the issue is we need to normalize the data before plotting it, let's try with log transformation

In [ ]:
# plot reviews and installs in a scatter plot
plt.figure(figsize=(16, 6)) # make figure size
sns.scatterplot(x=np.log10(df['Reviews']), y=np.log10(df['Installs']), data=df) # plot the scatter plot

In [ ]:
# Now we see a slight trend but still the issue is installs were given in a factorial manner, as 10+, 20+, 1000+ etc, and these are not continuous number but Discreet one, therefore, we can only see a slight trends here. Let's plot a line plot to see the trend.

In [ ]:
# plot reviews and installs in a scatter plot with trend line
plt.figure(figsize=(16, 6)) # make figure size
sns.lmplot(x='Reviews', y='Installs', data=df) # plot the scatter plot with trend line

In [ ]:
# find duplicate if any
df.duplicated().sum()

In [ ]:
# let's check for number of duplicates
for col in df.columns:
    print(f"Number of duplicates in {col} column are: {df[col].duplicated().sum()}")

In [ ]:
# print the number of duplicates in df
print(f"Number of duplicates in df are: {df.duplicated().sum()}")

## 9. Duplicate Removal and Final Dataset Checks

In [ ]:
# remove the duplicates
df.drop_duplicates(inplace=True)

In [ ]:
#  Insights from Data

## 10. Category-Level Analysis

In [ ]:
# 3.1. Which category has the highest number of apps?
df['Category'].value_counts().head(10) # this will show the top 10 categories with highest number of apps

In [ ]:
# Which category has the highest number of installs?
# category with highest number of Installs
df.groupby('Category')['Installs'].sum().sort_values(ascending=False).head(10)

In [ ]:
# Which category has the highest number of reviews?
df.groupby('Category')['Reviews'].sum().sort_values(ascending=False).head(10)

In [ ]:
#  Which category has the highest rating?
df.groupby('Category')['Rating'].mean().sort_values(ascending=False).head(10)

## 11. Rating Distribution

In [ ]:
# plot the rating distribution
plt.figure(figsize=(16, 6)) # make figure size
sns.kdeplot(df['Rating'], shade=True) # plot the distribution plot